**Now we move from Bronze → Silver.**

**This is where we perform the actual data-quality cleanup and deduplication.**

```text
BRONZE
  │
  ├── NULL validation
  ├── Invalid quantity
  ├── Invalid price
  ├── Status standardization
  └── Duplicate removal
          ↓
       SILVER
```

**So our actions should be:**

```text
Problem                    Action
------------------------------------------------
NULL customer_id           Remove
quantity <= 0              Remove
unit_price < 0             Remove
" completed "              Standardize
Duplicate order_id         Keep latest record
```

### Create Silver schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.silver;

### Create Silver table
**We'll use a temporary view first so the transformation is easy to understand.**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cleaned_orders AS

SELECT
    order_id,
    customer_id,
    order_date,
    product_id,
    quantity,
    unit_price,

    UPPER(TRIM(status)) AS status,

    ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY order_date DESC
    ) AS rn

FROM retail_lakehouse.bronze.orders

WHERE order_id IS NOT NULL
  AND customer_id IS NOT NULL
  AND order_date IS NOT NULL
  AND product_id IS NOT NULL
  AND quantity > 0
  AND unit_price >= 0;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.silver.orders
USING DELTA
AS

SELECT
    order_id,
    customer_id,
    order_date,
    product_id,
    quantity,
    unit_price,
    status

FROM cleaned_orders

WHERE rn = 1;

num_affected_rows,num_inserted_rows


**That's the core Silver logic.**

### Validate Silver Table

In [0]:
%sql
SELECT COUNT(*) AS silver_records
FROM retail_lakehouse.silver.orders;

silver_records
13713


In [0]:
%sql
-- Remaining Invalid Records
SELECT COUNT(*) AS invalid_records
FROM retail_lakehouse.silver.orders
WHERE customer_id IS NULL
   OR quantity <= 0
   OR unit_price < 0;

invalid_records
0


### Show Duplicates Removed

In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS cnt
FROM retail_lakehouse.silver.orders
GROUP BY order_id
HAVING COUNT(*) > 1;

order_id,cnt


### Check standardized status

In [0]:
%sql
SELECT DISTINCT status
FROM retail_lakehouse.silver.orders;

status
COMPLETED
RETURNED
CANCELLED
